In [77]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.12.0.dev20260217+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5080


In [78]:
import re
import numpy as np
import napari
from pathlib import Path
from typing import Optional
from magicgui import magicgui
from magicgui.widgets import Container, TextEdit
from liffile import LifFile
from pixel_size_utils import read_voxel_size_um, um_to_xy_pixels
from z_vote_utils import select_single_vote_and_down, contiguous_min_vote_range
from skimage import util
from skimage.filters import threshold_triangle, median, sobel, gaussian, threshold_yen
from skimage.measure import label, regionprops_table, regionprops, moments_central
from skimage.morphology import disk, remove_small_objects, remove_small_holes, erosion, closing
from skimage.transform import ProjectiveTransform, warp, probabilistic_hough_line
from scipy.ndimage import rotate, binary_dilation
from skimage.draw import line
import tifffile


In [79]:
class DeviceSegmentationApp:
    def __init__(
        self,
        low_frac: float = 0.82,
        high_frac: float = 1.10,
        smooth_window: int = 5,
        bin_size: float = 2.0,
        min_run_frac: float = 0.25,
        typical_pct: float = 50.0,
        line_length: int = 400,
        line_gap: int = 900,
        hough_threshold: int = 70,
        mask_sigma: float = 2.0,
        mask_frac_thresh: float = 0.70,
    ):
        self.low_frac = low_frac
        self.high_frac = high_frac
        self.smooth_window = smooth_window
        self.bin_size = bin_size
        self.min_run_frac = min_run_frac
        self.typical_pct = typical_pct
        self.line_length = line_length
        self.line_gap = line_gap
        self.hough_threshold = hough_threshold
        self.mask_sigma = mask_sigma
        self.mask_frac_thresh = mask_frac_thresh

        self.viewer = napari.Viewer()

        self._selected_image_folder: Optional[Path] = None
        self._selected_lif: Optional[Path] = None
        self._image_paths = []
        self._image_choice_map = {}

        self._roi_layer = None
        self._roi_outer_layer = None
        self._active_device_width_um = None
        self._syncing_outer_geometry = False
        self._cropped_layer = None
        self._last_image = None
        self._last_image_path = None
        self._last_see_interim_layers = True

        self._last_stack = None
        self._last_focus_downsample = 1
        self._last_focus_n_sampling = 10
        self._last_focus_patch = 50

        self._last_z_step_um = None
        self._last_y_step_um = None
        self._last_x_step_um = None
        self._last_xy_step_um = None
        self._last_geometry_vote_counts = None
        self._loaded_voxel_um = (None, None, None)

        self._cropped_stack_xy_raw = None
        self._cropped_stack_z_raw = None
        self.cropped_xyz = None


        self.images_output = TextEdit(value="")
        try:
            self.images_output.native.setReadOnly(True)
        except Exception:
            pass
        self.images_output.min_height = 120
        self.images_output.max_height = 300

        @magicgui(
            image_source={"label": "Folder/.tif/.lif", "mode": "r"},
            call_button="Load images",
        )
        def list_images(image_source = Path()):
            self._list_images(image_source)

        @magicgui(
            image_choice={"label": "Image", "choices": ["(load images)"], "widget_type": "ComboBox"},
            mask_central_region={"label": "Mask central region"},
            see_interim_layers={"label": "See interim layers (debug)"},
            clear_layers={"label": "Clear viewer first"},
            call_button="Segment + View",
        )
        def segment_and_view(image_choice: str = "(load images)",
            mask_central_region: bool = False,
            see_interim_layers: bool = False,
            clear_layers: bool = True,
        ):
            self._segment_and_view(
                image_choice=image_choice,
                focus_downsample=4,
                focus_n_sampling=10,
                focus_patch=50,
                mask_central_region=mask_central_region,
                see_interim_layers=see_interim_layers,
                clear_layers=clear_layers,
            )

        @magicgui(call_button="Create cropped aligned")
        def apply_crop():
            self._apply_crop_from_roi()

        @magicgui(
            auto_call=True,
            call_button=False,
            device_width_um={"label": "Device width (um)", "min": 0.0, "max": 1000000.0, "step": 1.0},
        )
        def device_width_ok(device_width_um: float = 0.0):
            self._apply_device_width_layer(device_width_um)

        self.list_images = list_images
        self.segment_and_view = segment_and_view
        self.apply_crop = apply_crop
        self.device_width_ok = device_width_ok


        self.main_panel = Container(
            widgets=[
                self.list_images,
                self.segment_and_view,
                self.device_width_ok,
                self.apply_crop,
                self.images_output,
            ]
        )
        self.viewer.window.add_dock_widget(self.main_panel, area="right")

        self._update_segment_button()

    def _fmt_um(self, value):
        if value is None:
            return "NA"
        try:
            vf = float(value)
        except Exception:
            return "NA"
        if not np.isfinite(vf):
            return "NA"
        return f"{vf:.4g}"

    def _voxel_log_text(self, z_um, y_um, x_um):
        return f"Voxel size (um): x={self._fmt_um(x_um)}, y={self._fmt_um(y_um)}, z={self._fmt_um(z_um)}"

    # -------- Focus helpers (integrated; no separate class) --------
    def _to_gray(self, im):
        if im.ndim == 2:
            return im
        return np.mean(im, axis=-1)

    def _focus_score(self, patch):
        p = np.asarray(self._to_gray(patch), dtype=float)
        return float(np.std(sobel(p)))

    def _curved_plane_refocus(self, stack_zyx, grid=20, patch=50, mask=None):
        Z, H, W = stack_zyx.shape
        ys = np.linspace(patch // 2, H - patch // 2 - 1, grid).astype(int)
        xs = np.linspace(patch // 2, W - patch // 2 - 1, grid).astype(int)

        pts, zs = [], []
        for y in ys:
            for x in xs:
                if mask is not None and not mask[y, x]:
                    continue
                sl = (slice(y - patch // 2, y + patch // 2), slice(x - patch // 2, x + patch // 2))
                f = np.array([self._focus_score(stack_zyx[z][sl]) for z in range(Z)], dtype=np.float32)
                pts.append((x, y))
                zs.append(int(np.argmax(f)))

        if len(zs) < 6:
            scores = []
            for z in range(Z):
                sl = self._to_gray(stack_zyx[z])
                if mask is not None:
                    sl = sl * mask
                scores.append(np.std(sobel(sl)))
            z0 = int(np.argmax(scores))
            img = self._to_gray(stack_zyx[z0])
            zmap = np.full(stack_zyx.shape[1:], z0, dtype=np.int16)
            return img, zmap, np.array(pts), np.array(zs)

        pts = np.array(pts, dtype=np.float32)
        zs = np.array(zs, dtype=np.float32)

        Xc = pts[:, 0] - pts[:, 0].mean()
        Yc = pts[:, 1] - pts[:, 1].mean()
        scale = max(W, H)
        Xn = Xc / scale
        Yn = Yc / scale

        B = np.column_stack((Xn**2, Yn**2, Xn * Yn, Xn, Yn, np.ones_like(Xn)))
        coeffs, *_ = np.linalg.lstsq(B, zs, rcond=None)

        Xg, Yg = np.meshgrid(np.arange(W), np.arange(H))
        mean_x = pts[:, 0].mean()
        mean_y = pts[:, 1].mean()
        Xg_n = (Xg - mean_x) / scale
        Yg_n = (Yg - mean_y) / scale
        zmap = (
            coeffs[0] * Xg_n**2
            + coeffs[1] * Yg_n**2
            + coeffs[2] * Xg_n * Yg_n
            + coeffs[3] * Xg_n
            + coeffs[4] * Yg_n
            + coeffs[5]
        )
        zmap = np.clip(np.rint(zmap).astype(np.int16), 0, Z - 1)
        img = np.take_along_axis(stack_zyx, zmap[None, :, :], axis=0)[0]
        return img, zmap, pts, zs
    def _compute_focus_plane_from_stack(self, stack: Optional[np.ndarray], downsample: int, n_sampling: int, patch: int, source_is_lif: bool = False, image_index: Optional[int] = None):
        # Unified focus path: if LIF, load + normalize here; elif use provided stack.
        if source_is_lif:
            if self._selected_lif is None:
                raise ValueError("No LIF file selected")
            with LifFile(self._selected_lif) as lif:
                idx = int(image_index if image_index is not None else 0)
                img = lif.images[idx]
                image = img.asarray()
            arr = np.asarray(image)
            if arr.ndim == 2:
                arr = arr[np.newaxis, ...]
            elif arr.ndim == 3:
                arr = arr if arr.shape[0] < 64 else arr[np.newaxis, ...]
            elif arr.ndim != 4:
                raise ValueError(f"Unsupported LIF array shape: {arr.shape}")

            self._last_stack = arr.astype(np.float32)
            z_um, y_um, x_um = self._read_voxel_size_um(self._selected_lif, source_is_lif=True, image_index=idx)
            self._last_z_step_um = z_um
            self._last_y_step_um = y_um
            self._last_x_step_um = x_um
            self._last_xy_step_um = np.nanmean([v for v in [x_um, y_um] if v is not None]) if (x_um is not None or y_um is not None) else None
        else:
            arr = np.asarray(stack)

        nz = int(arr.shape[0])
        ds = max(1, int(downsample))
        self._last_focus_downsample = ds

        stack_score_full = np.mean(arr, axis=-1) if arr.ndim == 4 else arr
        focus_full, _, _, zs = self._curved_plane_refocus(stack_score_full, grid=int(n_sampling), patch=int(patch), mask=None)

        if ds > 1:
            focus_out = focus_full[::ds, ::ds]
        else:
            focus_out = focus_full

        return focus_out

    # -------- segmentation + geometry --------
    def _is_color_image_2d(self, arr: np.ndarray) -> bool:
        return arr.ndim == 3 and arr.shape[-1] in (3, 4) and arr.shape[0] > 32 and arr.shape[1] > 32

    def _scale_to_uint8_view(self, arr):
        if arr is None:
            return None
        data = np.asarray(arr)
        dmin = float(np.nanmin(data))
        dmax = float(np.nanmax(data))
        if not np.isfinite(dmin) or not np.isfinite(dmax) or dmax <= dmin:
            return np.zeros_like(data, dtype=np.uint8)
        scaled = np.clip((data - dmin) / (dmax - dmin), 0.0, 1.0)
        return (scaled * 255.0).astype(np.uint8)

    def _order_corners_clockwise(self, corners_xy: np.ndarray):
        c = np.asarray(corners_xy, dtype=float)
        centroid = c.mean(axis=0)
        angles = np.arctan2(c[:, 1] - centroid[1], c[:, 0] - centroid[0])
        c = c[np.argsort(angles)]
        start = np.argmin(c[:, 0] + c[:, 1])
        return np.roll(c, -start, axis=0)

    def _crop_rectified_from_corners(self, in_focus_plane: np.ndarray, corners_xy: np.ndarray):
        if corners_xy is None:
            return None
        c = self._order_corners_clockwise(corners_xy)
        w1 = np.hypot(c[1, 0] - c[0, 0], c[1, 1] - c[0, 1])
        w2 = np.hypot(c[2, 0] - c[3, 0], c[2, 1] - c[3, 1])
        h1 = np.hypot(c[3, 0] - c[0, 0], c[3, 1] - c[0, 1])
        h2 = np.hypot(c[2, 0] - c[1, 0], c[2, 1] - c[1, 1])
        width = int(max(w1, w2))
        height = int(max(h1, h2))
        if width <= 1 or height <= 1:
            return None

        dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=float)
        tform = ProjectiveTransform()
        if not tform.estimate(dst, c):
            return None
        warped = warp(in_focus_plane, tform, output_shape=(height, width), order=1, preserve_range=True)
        return warped.astype(in_focus_plane.dtype)

    def _crop_rectified_stack_from_corners(self, stack: np.ndarray, corners_xy: np.ndarray):
        if corners_xy is None or stack is None:
            return None
        c = self._order_corners_clockwise(corners_xy)
        w1 = np.hypot(c[1, 0] - c[0, 0], c[1, 1] - c[0, 1])
        w2 = np.hypot(c[2, 0] - c[3, 0], c[2, 1] - c[3, 1])
        h1 = np.hypot(c[3, 0] - c[0, 0], c[3, 1] - c[0, 1])
        h2 = np.hypot(c[2, 0] - c[1, 0], c[2, 1] - c[1, 1])
        width = int(max(w1, w2))
        height = int(max(h1, h2))
        if width <= 1 or height <= 1:
            return None

        dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=float)
        tform = ProjectiveTransform()
        if not tform.estimate(dst, c):
            return None

        if stack.ndim == 3:
            warped = np.stack(
                [warp(stack[z], tform, output_shape=(height, width), order=1, preserve_range=True) for z in range(stack.shape[0])],
                axis=0,
            )
        elif stack.ndim == 4:
            warped = np.stack(
                [
                    warp(
                        stack[z],
                        tform,
                        output_shape=(height, width),
                        order=1,
                        preserve_range=True,
                        channel_axis=-1,
                    )
                    for z in range(stack.shape[0])
                ],
                axis=0,
            )
        else:
            return None

        return warped.astype(stack.dtype)

    def _signed_orientation(self, region):
        img = region.image.astype(float)
        mu = moments_central(img)
        angle_rad = 0.5 * np.arctan2(2 * mu[1, 1], mu[2, 0] - mu[0, 2])
        return np.rad2deg(angle_rad)

    def _corners_touch_border(self, corners_xy: np.ndarray, shape, margin=0):
        H, W = shape
        x = corners_xy[:, 0]
        y = corners_xy[:, 1]
        return (x <= margin).any() or (x >= (W - 1 - margin)).any() or (y <= margin).any() or (y >= (H - 1 - margin)).any()

    def _mask_out_organoid(self, in_focus_plane):
        inverted = gaussian(
            util.invert(np.asarray(in_focus_plane, dtype=np.float32)),
            sigma=float(self.mask_sigma),
            mode="nearest",
            preserve_range=True,
        ).astype(np.float32)
        H, W = in_focus_plane.shape[:2]
        xy_area = H * W
        cyi, cxi = H // 2, W // 2
        r = int(min(H, W) * 0.1)
        yy, xx = np.ogrid[:H, :W]
        central_roi = (yy - cyi) ** 2 + (xx - cxi) ** 2 <= r**2

        thresh = threshold_yen(inverted)
        labelled = label(inverted > thresh)
        props = regionprops(labelled)
        if len(props) == 0:
            return np.zeros((H, W), dtype=bool)

        def score(p):
            overlap = np.sum(labelled[central_roi] == p.label)
            py, px = p.centroid
            dist2 = (py - cyi) ** 2 + (px - cxi) ** 2
            return (-overlap, dist2)

        best_prop = min(props, key=score)
        if (best_prop.area > xy_area * self.mask_frac_thresh) or (best_prop.solidity < 0.5):
            thresh = threshold_triangle(inverted)
            labelled = label(inverted > thresh)
            props = regionprops(labelled)
            if len(props) == 0:
                return np.zeros((H, W), dtype=bool)
            best_prop = min(props, key=score)

        organoid_region = labelled == best_prop.label
        organoid_region = remove_small_holes(organoid_region, area_threshold=50000)
        organoid_region = closing(organoid_region, disk(5))
        return organoid_region

    def _oriented_rect_corners_crop_necks_and_flares(self, mask: np.ndarray):
        ys, xs = np.nonzero(mask)
        if xs.size == 0:
            return None, None, None

        cx, cy = xs.mean(), ys.mean()
        centroid_xy = np.array([cx, cy], float)

        mu = moments_central(mask.astype(np.uint8))
        angle_rad = 0.5 * np.arctan2(2 * mu[1, 1], (mu[0, 2] - mu[2, 0]))

        c, s = np.cos(angle_rad), np.sin(angle_rad)
        dx = xs - cx
        dy = ys - cy
        u = c * dx + s * dy
        v = -s * dx + c * dy

        u_span = float(u.max() - u.min())
        v_span = float(v.max() - v.min())
        if v_span >= u_span:
            long, short, long_name = v, u, "v"
        else:
            long, short, long_name = u, v, "u"

        short_min_full = float(short.min())
        short_max_full = float(short.max())
        long_bin = np.floor(long / self.bin_size).astype(int)
        bins, inv = np.unique(long_bin, return_inverse=True)

        short_min = np.full(bins.shape, np.inf)
        short_max = np.full(bins.shape, -np.inf)
        np.minimum.at(short_min, inv, short)
        np.maximum.at(short_max, inv, short)
        widths = short_max - short_min

        if self.smooth_window and self.smooth_window > 1:
            w = int(self.smooth_window)
            if w % 2 == 0:
                w += 1
            pad = w // 2
            widths_s = np.convolve(np.pad(widths, (pad, pad), mode="edge"), np.ones(w) / w, mode="valid")
        else:
            widths_s = widths

        typical_width = float(np.percentile(widths_s, self.typical_pct))
        low_thr = self.low_frac * typical_width
        high_thr = self.high_frac * typical_width
        keep = (~(widths_s < low_thr)) & (~(widths_s > high_thr))
        too_wide = widths_s > high_thr

        best_start = best_end = None
        best_len = 0
        cur_start = cur_end = None
        for i in range(len(bins)):
            if not keep[i]:
                if cur_start is not None:
                    L = cur_end - cur_start + 1
                    if L > best_len:
                        best_len = L
                        best_start, best_end = cur_start, cur_end
                    cur_start = cur_end = None
                continue
            if cur_start is None:
                cur_start = cur_end = bins[i]
            elif bins[i] == cur_end + 1:
                cur_end = bins[i]
            else:
                L = cur_end - cur_start + 1
                if L > best_len:
                    best_len = L
                    best_start, best_end = cur_start, cur_end
                cur_start = cur_end = bins[i]

        if cur_start is not None:
            L = cur_end - cur_start + 1
            if L > best_len:
                best_len = L
                best_start, best_end = cur_start, cur_end

        total_len_bins = int(bins.max() - bins.min() + 1)
        min_run_bins = int(np.ceil(self.min_run_frac * total_len_bins))

        if best_start is None or best_len < min_run_bins:
            long_min = float(long.min())
            long_max = float(long.max())
            crop_width = False
        else:
            long_min = float(best_start * self.bin_size)
            long_max = float((best_end + 1) * self.bin_size)
            band_mask_bins = (bins >= best_start) & (bins <= best_end)
            crop_width = bool(np.any(too_wide & (~band_mask_bins)))

        in_long_band = (long >= long_min) & (long <= long_max)
        if crop_width:
            short_min_use = float(short[in_long_band].min())
            short_max_use = float(short[in_long_band].max())
        else:
            short_min_use = short_min_full
            short_max_use = short_max_full

        if long_name == "v":
            umin, umax = short_min_use, short_max_use
            vmin, vmax = long_min, long_max
        else:
            umin, umax = long_min, long_max
            vmin, vmax = short_min_use, short_max_use

        corners_uv = np.array([[umin, vmin], [umax, vmin], [umax, vmax], [umin, vmax]], dtype=float)
        x = cx + c * corners_uv[:, 0] - s * corners_uv[:, 1]
        y = cy + s * corners_uv[:, 0] + c * corners_uv[:, 1]
        return np.stack([x, y], axis=1), angle_rad, centroid_xy

    def _segment_from_plane(self, in_focus_plane: np.ndarray, mask_central_region: bool, return_debug: bool):
        flag = False
        in_focus_plane = self._to_gray(in_focus_plane)

        median_thresholded = median(np.asarray(in_focus_plane, dtype=np.float32), footprint=disk(7)).astype(np.float32)
        sobel_operated = sobel(median_thresholded).astype(np.float32)
        thresh = threshold_triangle(sobel_operated)
        binary = sobel_operated > thresh

        h, w = in_focus_plane.shape[:2]
        binary[h // 3 : 2 * (h // 3), w // 3 : 2 * (w // 3)] = 0

        organoid_region = None
        if mask_central_region:
            organoid_region = self._mask_out_organoid(in_focus_plane)
            binary[organoid_region] = 0

        labels = label(binary)
        data = regionprops_table(labels, binary, properties=("label", "area", "eccentricity"))
        condition = (data["area"] > 100) & (data["eccentricity"] > 0.5)
        labels_to_dilate = util.map_array(labels, data["label"], data["label"] * condition)

        dilated_output = np.zeros_like(labels, dtype=np.uint8)
        base_selem = np.zeros((31, 31), dtype=bool)
        base_selem[15, :] = 1
        pad = base_selem.shape[0] // 2

        for region in regionprops(labels_to_dilate):
            angle_to_rotate = self._signed_orientation(region)
            rotated_selem = rotate(base_selem.astype(float), angle=90 + angle_to_rotate, reshape=False, order=0) > 0.5

            minr, minc, maxr, maxc = region.bbox
            r0 = max(minr - pad, 0)
            r1 = min(maxr + pad, labels_to_dilate.shape[0])
            c0 = max(minc - pad, 0)
            c1 = min(maxc + pad, labels_to_dilate.shape[1])

            mask_roi = labels_to_dilate[r0:r1, c0:c1] == region.label
            dilated = binary_dilation(mask_roi.astype(bool), structure=rotated_selem.astype(bool))
            dilated_output[r0:r1, c0:c1][dilated] = 255

        post_dilation_mask = np.logical_or(dilated_output, binary)
        clean_labels = label(~post_dilation_mask)
        props = regionprops(clean_labels)
        largest_prop = max(props, key=lambda p: p.area)
        device_mask = clean_labels == largest_prop.label

        edges = reconstructed = reconstructed_mask = None
        new_corners = new_angle_rad = new_centroid_xy = None

        corners, angle_rad, centroid_xy = self._oriented_rect_corners_crop_necks_and_flares(device_mask)
        if corners is None or self._corners_touch_border(corners, device_mask.shape, margin=5):
            flag = True
            edges = remove_small_objects(labels_to_dilate > 0)
            segs = probabilistic_hough_line(
                edges,
                line_length=self.line_length,
                line_gap=self.line_gap,
                threshold=self.hough_threshold,
            )
            reconstructed = np.zeros_like(edges, dtype=bool)
            for (x0, y0), (x1, y1) in segs:
                rr, cc = line(y0, x0, y1, x1)
                reconstructed[rr, cc] = True

            reconstructed_mask = np.logical_or(reconstructed, post_dilation_mask)
            updated_clean_labels = label(~reconstructed_mask)
            props = regionprops(updated_clean_labels)
            largest_prop = max(props, key=lambda p: p.area)
            new_device_mask = updated_clean_labels == largest_prop.label
            new_corners, new_angle_rad, new_centroid_xy = self._oriented_rect_corners_crop_necks_and_flares(new_device_mask)

        if flag:
            final_corners = new_corners
            final_angle_rad = new_angle_rad
            final_centroid_xy = new_centroid_xy
        else:
            final_corners = corners
            final_angle_rad = angle_rad
            final_centroid_xy = centroid_xy

        cropped_rotated = self._crop_rectified_from_corners(in_focus_plane, final_corners)

        if return_debug:
            debug = {
                "median_thresholded": median_thresholded,
                "sobel_operated": sobel_operated,
                "binary": binary,
                "labels_to_dilate": labels_to_dilate,
                "post_dilation_mask": post_dilation_mask,
                "device_mask": device_mask,
                "organoid_region": organoid_region,
                "edges": edges,
                "reconstructed": reconstructed,
                "reconstructed_mask": reconstructed_mask,
                "flag": flag,
                "final_corners": final_corners,
                "final_centroid_xy": final_centroid_xy,
                "final_angle_rad": final_angle_rad,
                "new_corners": new_corners,
                "new_centroid_xy": new_centroid_xy,
                "new_angle_rad": new_angle_rad,
                "cropped_rotated": cropped_rotated,
                "gpu_used_preprocess": False,
                "gpu_used_dilation": False,
            }
            return in_focus_plane, organoid_region, final_corners, cropped_rotated, debug

        return in_focus_plane, organoid_region, final_corners, cropped_rotated

    # -------- IO / GUI flow --------
    def _list_images(self, image_source: Path):
        p = Path(image_source)
        if not p.exists():
            self.images_output.value = "[WARN] Please select a folder, a .tif/.tiff file, or a .lif file."
            return

        self._selected_image_folder = None
        self._selected_lif = None
        self._image_paths = []
        self._last_stack = None
        self._last_focus_downsample = 1
        choices = []
        choice_map = {}

        if p.is_dir():
            self._selected_image_folder = p
            image_files = sorted([q for q in p.iterdir() if q.is_file() and q.suffix.lower() in (".tif", ".tiff")])
            if not image_files:
                self.images_output.value = "[WARN] No .tif/.tiff images found in the selected folder."
                self._image_choice_map = {}
                self.segment_and_view.image_choice.choices = ["(load images)"]
                self.segment_and_view.image_choice.value = "(load images)"
                self._update_segment_button()
                return
            self._image_paths = image_files
            for i, pth in enumerate(image_files):
                label_txt = f"{i}: {pth.name}"
                choices.append(label_txt)
                choice_map[label_txt] = i
            self._image_choice_map = choice_map
            self.segment_and_view.image_choice.choices = choices
            self.segment_and_view.image_choice.value = choices[0]
            z_um, y_um, x_um = self._read_voxel_size_um(image_files[0], source_is_lif=False)
            self._loaded_voxel_um = (z_um, y_um, x_um)
            self.images_output.value = f"[OK] Found {len(choices)} images in folder. Select one and click Segment + View. {self._voxel_log_text(z_um, y_um, x_um)}"
            self._update_segment_button()
            return

        if p.suffix.lower() == ".lif":
            self._selected_lif = p
            try:
                with LifFile(self._selected_lif) as lif:
                    for i, img in enumerate(lif.images):
                        name = img.name if hasattr(img, "name") and img.name else f"image_{i}"
                        label_txt = f"{i}: {name}"
                        choices.append(label_txt)
                        choice_map[label_txt] = i
            except Exception:
                self.images_output.value = "[ERROR] Unable to read .lif contents."
                self._image_choice_map = {}
                self.segment_and_view.image_choice.choices = ["(load images)"]
                self.segment_and_view.image_choice.value = "(load images)"
                self._update_segment_button()
                return

            if not choices:
                self.images_output.value = "[WARN] No readable images found inside the selected .lif file."
                self._image_choice_map = {}
                self.segment_and_view.image_choice.choices = ["(load images)"]
                self.segment_and_view.image_choice.value = "(load images)"
                self._update_segment_button()
                return

            self._image_choice_map = choice_map
            self.segment_and_view.image_choice.choices = choices
            self.segment_and_view.image_choice.value = choices[0]
            first_idx = choice_map[choices[0]]
            z_um, y_um, x_um = self._read_voxel_size_um(self._selected_lif, source_is_lif=True, image_index=first_idx)
            self._loaded_voxel_um = (z_um, y_um, x_um)
            self.images_output.value = f"[OK] Found {len(choices)} images in .lif. Select one and click Segment + View. {self._voxel_log_text(z_um, y_um, x_um)}"
            self._update_segment_button()
            return

        if p.suffix.lower() in (".tif", ".tiff"):
            self._image_paths = [p]
            label_txt = f"0: {p.name}"
            self._image_choice_map = {label_txt: 0}
            self.segment_and_view.image_choice.choices = [label_txt]
            self.segment_and_view.image_choice.value = label_txt
            z_um, y_um, x_um = self._read_voxel_size_um(p, source_is_lif=False)
            self._loaded_voxel_um = (z_um, y_um, x_um)
            self.images_output.value = f"[OK] Loaded single .tif file. Select it and click Segment + View. {self._voxel_log_text(z_um, y_um, x_um)}"
            self._update_segment_button()
            return

        self.images_output.value = "[WARN] Unsupported selection. Choose a folder, a .tif/.tiff file, or a .lif file."
        self._image_choice_map = {}
        self.segment_and_view.image_choice.choices = ["(load images)"]
        self.segment_and_view.image_choice.value = "(load images)"
        self._update_segment_button()

    def _segment_and_view(
        self,
        image_choice: str,
        focus_downsample: int,
        focus_n_sampling: int,
        focus_patch: int,
        mask_central_region: bool,
        see_interim_layers: bool,
        clear_layers: bool,
    ):
        source_is_lif = bool(getattr(self, "_selected_lif", None) and self._selected_lif.exists())

        if not source_is_lif and not self._image_paths:
            self.images_output.value = "[WARN] Select a folder/.tif/.lif and click 'Load images' first."
            return
        if not self._image_choice_map:
            self.images_output.value = "[WARN] Click 'Load images' to populate the dropdown."
            return

        image_index = self._image_choice_map.get(image_choice)
        if image_index is None:
            self.images_output.value = "[WARN] Please select an image from the dropdown."
            return

        self._last_focus_n_sampling = int(focus_n_sampling)
        self._last_focus_patch = int(focus_patch)

        try:
            if source_is_lif:
                in_focus_plane = self._compute_focus_plane_from_stack(
                    stack=None,
                    downsample=focus_downsample,
                    n_sampling=focus_n_sampling,
                    patch=focus_patch,
                    source_is_lif=True,
                    image_index=image_index,
                )
                in_focus_plane, organoid_region, final_corners, cropped_rotated, debug = self._segment_from_plane(
                    in_focus_plane,
                    mask_central_region,
                    return_debug=True,
                )
                self._last_image_path = self._selected_lif
            else:
                source_path = self._image_paths[image_index]
                arr = np.asarray(tifffile.imread(str(source_path)))

                z_um, y_um, x_um = self._read_voxel_size_um(Path(source_path), source_is_lif=False)
                self._last_z_step_um = z_um
                self._last_y_step_um = y_um
                self._last_x_step_um = x_um
                self._last_xy_step_um = np.nanmean([v for v in [x_um, y_um] if v is not None]) if (x_um is not None or y_um is not None) else None

                if self._is_color_image_2d(arr) or arr.ndim < 3:
                    in_focus_plane = arr.astype(np.float32)
                    self._last_stack = None
                    self._last_focus_downsample = 1
                else:
                    stack = arr.astype(np.float32)
                    in_focus_plane = self._compute_focus_plane_from_stack(stack, focus_downsample, focus_n_sampling, focus_patch)
                    self._last_stack = stack
                    self._last_focus_downsample = max(1, int(focus_downsample))

                in_focus_plane, organoid_region, final_corners, cropped_rotated, debug = self._segment_from_plane(
                    in_focus_plane,
                    mask_central_region,
                    return_debug=True,
                )
                self._last_image_path = source_path
        except Exception as e:
            self.images_output.value = f"[ERROR] Segmentation failed: {type(e).__name__}: {e}"
            return

        if clear_layers:
            self.viewer.layers.clear()
            self._roi_layer = None
            self._roi_outer_layer = None
            self._cropped_layer = None

        self._last_image = in_focus_plane
        self._last_see_interim_layers = see_interim_layers
        self._active_device_width_um = 30.0
        self._last_geometry_vote_counts = None
        if self._roi_outer_layer is not None and self._roi_outer_layer in self.viewer.layers:
            self.viewer.layers.remove(self._roi_outer_layer)
        self._roi_outer_layer = None
        self._cropped_stack_xy_raw = None
        self._cropped_stack_z_raw = None
        self.cropped_xyz = None
        self.device_width_ok.device_width_um.enabled = True
        self.device_width_ok.device_width_um.value = 30.0


        self._add_layer_if_nonzero(in_focus_plane, name="original", layer_type="image")

        if see_interim_layers:
            self._add_layer_if_nonzero(debug.get("sobel_operated"), name="sobel", layer_type="image")
            self._add_layer_if_nonzero(debug.get("binary").astype(np.uint8), name="binary", layer_type="image")
            self._add_layer_if_nonzero(debug.get("labels_to_dilate").astype(np.int32), name="labels_to_dilate", layer_type="labels")
            final_layer = self._add_layer_if_nonzero(debug.get("post_dilation_mask").astype(np.uint8), name="post_dilation_mask", layer_type="labels")
            if final_layer is not None:
                final_layer.opacity = 0.4
            device_layer = self._add_layer_if_nonzero(debug.get("device_mask").astype(np.uint8), name="device_mask", layer_type="labels")
            if device_layer is not None:
                device_layer.opacity = 0.4
            if organoid_region is not None:
                organoid_layer = self._add_layer_if_nonzero(organoid_region.astype(np.uint8), name="organoid_region", layer_type="labels")
                if organoid_layer is not None:
                    organoid_layer.opacity = 0.4

        force_roi = clear_layers or self._roi_layer is None or len(getattr(self._roi_layer, "data", [])) == 0
        self._set_roi_layer(final_corners, force=force_roi)
        self._update_outer_geometry_from_current_roi(30.0, update_message=False)
        self.images_output.value = (
            "[OK] Segmentation complete. Outer geometry is shown at default Device width=30 um; adjust width as needed."
        )


    def _set_roi_layer(self, corners_xy, force=False):
        if corners_xy is None:
            self.images_output.value = "[WARN] No corners found for ROI."
            return
        corners_yx = np.asarray(corners_xy)[:, ::-1]
        if self._roi_layer is None or self._roi_layer not in self.viewer.layers:
            self._roi_layer = self.viewer.add_shapes(name="geometry")
            try:
                self._roi_layer.events.data.connect(self._on_roi_layer_data_changed)
            except Exception:
                pass
        if force or len(self._roi_layer.data) == 0:
            self._roi_layer.data = [corners_yx]
        self._roi_layer.shape_type = ["rectangle"]
        self._roi_layer.edge_color = "red"
        self._roi_layer.face_color = "transparent"
        self._roi_layer.editable = True
        self._roi_layer.mode = "select"

    def _on_roi_layer_data_changed(self, event=None):
        if self._syncing_outer_geometry:
            return
        if self._roi_outer_layer is None or self._roi_outer_layer not in self.viewer.layers:
            return

        width_um = self._active_device_width_um
        if width_um is None:
            try:
                width_um = float(self.device_width_ok.device_width_um.value)
            except Exception:
                return

        if not np.isfinite(width_um) or width_um < 0:
            return

        try:
            self._syncing_outer_geometry = True
            self._update_outer_geometry_from_current_roi(width_um, update_message=False)
        finally:
            self._syncing_outer_geometry = False

    def _get_current_roi_corners_xy(self):
        if self._roi_layer is None or len(self._roi_layer.data) == 0:
            return None
        corners_xy = np.asarray(self._roi_layer.data[0])[:, ::-1]
        return self._order_corners_clockwise(corners_xy)

    def _get_current_crop_corners_xy(self):
        if self._roi_outer_layer is not None and self._roi_outer_layer in self.viewer.layers:
            if len(getattr(self._roi_outer_layer, "data", [])) > 0:
                outer_xy = np.asarray(self._roi_outer_layer.data[0])[:, ::-1]
                return self._order_corners_clockwise(outer_xy)
        return self._get_current_roi_corners_xy()

    def _read_voxel_size_um(self, source_path: Path, source_is_lif: bool = False, image_index: Optional[int] = None):
        return read_voxel_size_um(source_path, source_is_lif=source_is_lif, image_index=image_index)

    def _um_to_xy_pixels(self, width_um: float):
        return um_to_xy_pixels(width_um, self._last_x_step_um, self._last_y_step_um)

    def _expand_rectangle_corners(self, corners_xy: np.ndarray, expand_x_px: float, expand_y_px: float):
        c = self._order_corners_clockwise(corners_xy)
        center = c.mean(axis=0)

        e0 = c[1] - c[0]
        e1 = c[3] - c[0]
        l0 = float(np.linalg.norm(e0))
        l1 = float(np.linalg.norm(e1))
        if l0 <= 0 or l1 <= 0:
            return None

        u0 = e0 / l0
        u1 = e1 / l1
        h0 = l0 * 0.5
        h1 = l1 * 0.5

        expanded = np.array(
            [
                center - (h0 + expand_x_px) * u0 - (h1 + expand_y_px) * u1,
                center + (h0 + expand_x_px) * u0 - (h1 + expand_y_px) * u1,
                center + (h0 + expand_x_px) * u0 + (h1 + expand_y_px) * u1,
                center - (h0 + expand_x_px) * u0 + (h1 + expand_y_px) * u1,
            ],
            dtype=float,
        )
        return expanded

    def _update_outer_geometry_from_current_roi(self, device_width_um: float, update_message: bool = True):
        corners_xy = self._get_current_roi_corners_xy()
        if corners_xy is None:
            if update_message:
                self.images_output.value = "[WARN] Geometry layer is missing. Run 'Segment + View' again."
            return False

        px_xy = self._um_to_xy_pixels(device_width_um)
        if px_xy is None:
            if update_message:
                self.images_output.value = (
                    "[WARN] Could not convert Device width (um) to pixels. Check image voxel metadata (x/y step)."
                )
            return False

        expand_x_px, expand_y_px = px_xy
        expanded_xy = self._expand_rectangle_corners(corners_xy, expand_x_px, expand_y_px)
        if expanded_xy is None:
            if update_message:
                self.images_output.value = "[WARN] Could not compute expanded geometry."
            return False

        expanded_yx = expanded_xy[:, ::-1]
        if self._roi_outer_layer is None or self._roi_outer_layer not in self.viewer.layers:
            self._roi_outer_layer = self.viewer.add_shapes(name="geometry_device_width")

        self._roi_outer_layer.data = [expanded_yx]
        self._roi_outer_layer.shape_type = ["rectangle"]
        self._roi_outer_layer.edge_color = "yellow"
        self._roi_outer_layer.face_color = "transparent"
        self._roi_outer_layer.editable = False

        if update_message:
            self.images_output.value = (
                f"[OK] Added outer geometry layer using Device width={float(device_width_um):.4g} um "
                f"(~dx={expand_x_px:.2f}px, dy={expand_y_px:.2f}px on each side)."
            )
        return True

    def _apply_device_width_layer(self, device_width_um: float):
        corners_xy = self._get_current_roi_corners_xy()
        if corners_xy is None:
            self.images_output.value = "[WARN] Draw or adjust rectangle first."
            return

        try:
            self._active_device_width_um = float(device_width_um)
        except Exception:
            self.images_output.value = "[WARN] Device width must be numeric."
            return

        self._update_outer_geometry_from_current_roi(self._active_device_width_um, update_message=True)

    def _compute_focus_patch_votes_for_stack(self, stack: np.ndarray, n_sampling: int, patch: int):
        if stack is None:
            return None, None

        stack_arr = np.asarray(stack)
        if stack_arr.ndim == 4:
            stack_gray = np.mean(stack_arr, axis=-1)
        elif stack_arr.ndim == 3:
            stack_gray = stack_arr
        else:
            return None, None

        Z, H, W = stack_gray.shape
        if Z <= 0 or H <= 0 or W <= 0:
            return None, None

        patch = max(5, int(patch))
        grid = max(4, int(n_sampling))
        half = patch // 2

        vote_volume = np.zeros((Z, H, W), dtype=np.float32)

        if H <= patch or W <= patch:
            scores = [np.std(sobel(self._to_gray(stack_gray[z]))) for z in range(Z)]
            z_best = int(np.argmax(scores))
            vote_volume[z_best, H // 2, W // 2] = 1.0
            counts = np.zeros(Z, dtype=np.int32)
            counts[z_best] = 1
            return counts, vote_volume

        ys = np.linspace(half, H - half - 1, grid).astype(int)
        xs = np.linspace(half, W - half - 1, grid).astype(int)

        counts = np.zeros(Z, dtype=np.int32)
        for y in ys:
            for x in xs:
                y0 = max(0, y - half)
                y1 = min(H, y + half)
                x0 = max(0, x - half)
                x1 = min(W, x + half)
                f = np.array([self._focus_score(stack_gray[z, y0:y1, x0:x1]) for z in range(Z)], dtype=np.float32)
                best_z = int(np.argmax(f))
                counts[best_z] += 1
                yy0 = max(0, y - 1)
                yy1 = min(H, y + 2)
                xx0 = max(0, x - 1)
                xx1 = min(W, x + 2)
                vote_volume[best_z, yy0:yy1, xx0:xx1] += 1.0

        return counts, vote_volume

    def _format_geometry_vote_planes(self):
        counts = self._last_geometry_vote_counts
        if counts is None:
            return "none"
        counts = np.asarray(counts).astype(int)
        if counts.size == 0:
            return "none"
        hits = np.flatnonzero(counts > 0)
        if hits.size == 0:
            return "none"
        return ", ".join([f"z{int(i)}:{int(counts[i])}" for i in hits])

    def _geometry_vote_summary_suffix(self):
        txt = self._format_geometry_vote_planes()
        return f" Geometry vote planes (all nonzero): {txt}."

    def _apply_crop_from_roi(self):
        corners_xy = self._get_current_crop_corners_xy()
        if corners_xy is None:
            self.images_output.value = "[WARN] Draw or adjust geometry first."
            return
        if self._last_image is None:
            self.images_output.value = "[WARN] Run 'Segment + View' first."
            return

        if self._last_stack is not None:
            scale = max(1, int(self._last_focus_downsample))
            cropped_stack = self._crop_rectified_stack_from_corners(self._last_stack, corners_xy * float(scale))
            if cropped_stack is None:
                self.images_output.value = "[WARN] Crop failed for selected geometry (stack)."
                return

            roi_inner_xy = self._get_current_roi_corners_xy()
            roi_stack_only = None
            if roi_inner_xy is not None:
                roi_stack_only = self._crop_rectified_stack_from_corners(self._last_stack, roi_inner_xy * float(scale))
            self._last_geometry_vote_counts = None

            self._cropped_stack_xy_raw = cropped_stack
            self._cropped_stack_z_raw = None
            self.cropped_xyz = None


            self._cropped_layer = self._add_or_update_image_layer(
                self._cropped_layer,
                self._scale_to_uint8_view(cropped_stack),
                "cropped_rotated",
            )
            counts = None
            if roi_stack_only is not None:
                counts, _ = self._compute_focus_patch_votes_for_stack(
                    roi_stack_only.astype(np.float32),
                    n_sampling=int(self._last_focus_n_sampling or 10),
                    patch=int(self._last_focus_patch or 50),
                )

            self._cropped_stack_z_raw = self._cropped_stack_xy_raw
            self.cropped_xyz = self._cropped_stack_z_raw

            if counts is not None and len(counts) > 0:
                self._last_geometry_vote_counts = np.asarray(counts, dtype=int)

                top_n = min(5, len(counts))
                top_idx = np.argsort(counts)[::-1][:top_n]
                top_txt = ", ".join([f"z{int(i)}:{int(counts[i])}" for i in top_idx])
                all_txt = self._format_geometry_vote_planes()
                per_layer_txt = ", ".join([f"z{int(i)}:{int(c)}" for i, c in enumerate(np.asarray(counts, dtype=int))])
                self.images_output.value = (
                    f"Geometry-only focus patch votes (inner geometry only; top planes): {top_txt}. "
                    f"All nonzero voted planes: {all_txt}. "
                    f"Votes for each layer: {per_layer_txt}."
                )
            else:
                self.images_output.value = (
                    "No geometry votes found."
                )
            return

        cropped_img = self._crop_rectified_from_corners(self._last_image, corners_xy)
        if cropped_img is None:
            self.images_output.value = "[WARN] Crop failed for selected geometry."
            return
        self.images_output.value = "[OK] Cropped aligned image created from current geometry."

    def get_cropped_outputs(self):
        """Return cropped data and geometry-vote metadata for downstream use."""
        z_um, y_um, x_um = self._loaded_voxel_um
        if self._last_z_step_um is not None:
            z_um = self._last_z_step_um
        if self._last_y_step_um is not None:
            y_um = self._last_y_step_um
        if self._last_x_step_um is not None:
            x_um = self._last_x_step_um

        xy_um = self._last_xy_step_um
        if xy_um is None and (x_um is not None or y_um is not None):
            xy_um = float(np.nanmean([v for v in [x_um, y_um] if v is not None]))

        z_votes = None
        if self._last_geometry_vote_counts is not None:
            counts = np.asarray(self._last_geometry_vote_counts, dtype=int)
            z_votes = {int(i): int(c) for i, c in enumerate(counts)}

        pixel_size_um = {
            "x_um": x_um,
            "y_um": y_um,
            "z_um": z_um,
            "xy_um": xy_um,
        }

        return self.cropped_xyz, self._active_device_width_um, pixel_size_um, z_votes

    # -------- viewer helpers --------
    def _add_or_update_image_layer(self, layer, data, name):
        if layer is not None and layer in self.viewer.layers:
            layer.data = data
            return layer
        return self._add_layer_if_nonzero(data, name=name, layer_type="image")

    def _has_signal(self, arr) -> bool:
        return arr is not None and np.any(arr)

    def _add_layer_if_nonzero(self, data, name, layer_type="image", **kwargs):
        if not self._has_signal(data):
            return None
        if layer_type == "labels":
            return self.viewer.add_labels(data, name=name, **kwargs)
        return self.viewer.add_image(data, name=name, **kwargs)

    def _update_segment_button(self, *_):
        self.segment_and_view.call_button.enabled = True


In [81]:

app = DeviceSegmentationApp()
viewer = app.viewer
segment_and_view = app.segment_and_view
list_images = app.list_images
images_output = app.images_output
main_panel = app.main_panel
get_cropped_outputs = app.get_cropped_outputs



cropped_xyz = app.cropped_xyz
print("cropped_xyz:", None if cropped_xyz is None else cropped_xyz.shape)

cropped_xyz: None


In [82]:
cropped_stack, device_width_um, pixel_size_um, z_votes = app.get_cropped_outputs()

print(type(cropped_stack), None if cropped_stack is None else cropped_stack.shape)
print("device_width_um:", device_width_um)
print("pixel_size_um:", pixel_size_um)   # {'x_um': ..., 'y_um': ..., 'z_um': ..., 'xy_um': ...}
print("z_votes sample:", None if z_votes is None else list(z_votes.items())[:10])

<class 'numpy.ndarray'> (50, 3703, 2313)
device_width_um: 35.0
pixel_size_um: {'x_um': 1.3000013000013, 'y_um': 1.3000013000013, 'z_um': 7.998110204081633, 'xy_um': np.float64(1.3000013000013)}
z_votes sample: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 1), (7, 1), (8, 7), (9, 14)]


In [44]:
viewer = napari.Viewer()
viewer.add_image(cropped_stack, name="cropped_stack")

<Image layer 'cropped_stack' at 0x2ab091b6f90>

In [74]:
def zs_single_vote_and_down(z_votes, min_span_um=160.0, z_step_um=None):
    """
    Select a z-range from vote counts and return votes limited to that range.

    Rules:
    1) Start from the largest contiguous span where vote >= 1.
    2) If its physical span is >= min_span_um, use it.
    3) If smaller, extend downward toward z=0 first.
    4) If z=0 is reached and still smaller, extend upward.

    Returns
    -------
    dict | None
        z-vote mapping limited to the selected z-range, in the same key style as input.
    """
    if z_step_um is None:
        z_step_um = (globals().get("pixel_size_um") or {}).get("z_um", None)
    if z_step_um is None and "app" in globals():
        z_step_um = getattr(app, "_last_z_step_um", None)
    if z_step_um is None:
        raise ValueError("z_step_um is required (pass it explicitly or ensure pixel metadata is loaded).")

    max_z = None
    if "cropped_stack" in globals() and isinstance(cropped_stack, np.ndarray) and cropped_stack.ndim >= 3:
        max_z = int(cropped_stack.shape[0] - 1)

    return select_single_vote_and_down(
        z_votes,
        min_span_um=float(min_span_um),
        z_step_um=float(z_step_um),
        max_z=max_z,
    )

In [75]:
def contiguous_double_vote_range(votes, z_step_um=None):
    """
    Return the largest contiguous z-band with vote >= 2, as a z_votes-style dict.
    """
    return contiguous_min_vote_range(votes, min_vote=2)

In [76]:
# Example usage:

selected_z_votes = zs_single_vote_and_down(z_votes)
selected_z_votes_ge2 = contiguous_double_vote_range(z_votes)
print("Original z_votes sample:", None if z_votes is None else list(z_votes.items()))
print("selected_z_votes sample:", None if selected_z_votes is None else list(selected_z_votes.items()))
print("selected_z_votes_ge2 sample:", None if selected_z_votes_ge2 is None else list(selected_z_votes_ge2.items()))
print("selected_range_len:", None if selected_z_votes is None else len(selected_z_votes))
print("selected_ge2_len:", None if selected_z_votes_ge2 is None else len(selected_z_votes_ge2))

Original z_votes sample: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 1), (8, 0), (9, 1), (10, 1), (11, 2), (12, 6), (13, 7), (14, 10), (15, 10), (16, 8), (17, 3), (18, 3), (19, 3), (20, 4), (21, 2), (22, 7), (23, 3), (24, 3), (25, 3), (26, 1), (27, 3), (28, 1), (29, 1), (30, 0), (31, 1), (32, 1), (33, 1), (34, 2), (35, 0), (36, 1), (37, 0), (38, 0), (39, 0), (40, 0), (41, 1), (42, 0), (43, 0), (44, 1), (45, 1), (46, 2), (47, 0), (48, 0), (49, 4), (50, 0), (51, 0), (52, 0), (53, 0), (54, 0), (55, 0), (56, 1), (57, 0), (58, 0), (59, 1), (60, 0), (61, 0), (62, 0), (63, 0), (64, 0), (65, 0)]
selected_z_votes sample: [(9, 1), (10, 1), (11, 2), (12, 6), (13, 7), (14, 10), (15, 10), (16, 8), (17, 3), (18, 3), (19, 3), (20, 4), (21, 2), (22, 7), (23, 3), (24, 3), (25, 3), (26, 1), (27, 3), (28, 1), (29, 1)]
selected_z_votes_ge2 sample: [(11, 2), (12, 6), (13, 7), (14, 10), (15, 10), (16, 8), (17, 3), (18, 3), (19, 3), (20, 4), (21, 2), (22, 7), (23, 3), (24, 3), (25, 3)]
selec

In [67]:
print(selected_z_votes)

{9: 1, 10: 1, 11: 2, 12: 6, 13: 7, 14: 10, 15: 10, 16: 8, 17: 3, 18: 3, 19: 3, 20: 4, 21: 2, 22: 7, 23: 3, 24: 3, 25: 3, 26: 1, 27: 3, 28: 1, 29: 1}
